In [1]:
from time import perf_counter  
from time import time
import numpy as np
import pandas as pd
from wombat import Simulation, load_yaml
from wombat import create_library_structure 
from time import perf_counter
from pathlib import Path
import yaml
from wombat.core import Metrics
import warnings 
warnings.filterwarnings("ignore")
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
from concurrent.futures import ProcessPoolExecutor

np.random.seed(3)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", 1000)

In [20]:
N_RUNS = 30
RANDOM_SEEDS = list(range(1, N_RUNS + 1))

In [ ]:
# new_library = "library"
# create_library_structure(new_library)

In [2]:
import os
print(os.listdir("library"))

['substations', 'turbines', 'weather', 'results', 'cables', 'vessels', 'project']


In [3]:
BASE_DIR = Path("library")

In [ ]:
# def run_windfarm_simulations(config_name, random_seeds: list):
#     results_dir = BASE_DIR / "results"
#     if not results_dir.is_dir():
#         results_dir.mkdir()

#     # Initialize storage lists
#     availability_records = []
#     opex_records = []
#     vessel_records = []
#     repair_time_records = []
#     power_records = []
#     fixed_cost_records = []
#     component_records = []

#     N = len(random_seeds)
#     for i, seed in enumerate(random_seeds, start=1):
#         print(
#             f" Running simulation {i}/N with random seed {seed}",
#             end="\r",
#         )

#         # Run simulation
#         sim = Simulation(BASE_DIR, config_name, random_seed=seed)
#         sim.run(create_metrics=True, save_metrics_inputs=True)

#         # Load metrics
#         fpath = sim.env.metrics_input_fname.parent
#         fname = sim.env.metrics_input_fname.name
#         metrics = Metrics.from_simulation_outputs(fpath, fname)

#         # Availability Results
#         time_avail = metrics.time_based_availability(frequency="project", by="windfarm")
#         prod_avail = metrics.production_based_availability(
#             frequency="project", by="windfarm"
#         )
#         time_value = time_avail.iloc[0, 0]
#         prod_value = prod_avail.iloc[0, 0]
#         availability_records.append(
#             {
#                 "run": i,
#                 "random_seed": seed,
#                 "time_based_availability": time_value,
#                 "production_based_availability": prod_value,
#             }
#         )

#         # OpEx Results 
#         opex_df = metrics.opex(frequency="annual", by_category=True).reset_index()
#         opex_df = opex_df.replace([np.inf, -np.inf], np.nan).dropna(how="any")
#         opex_df.insert(0, "random_seed", seed)
#         opex_df.insert(0, "run", i)
#         opex_records.append(opex_df)

#         # Vessel Costs 
#         vessel_df = metrics.equipment_costs(
#             frequency="annual", by_equipment=True
#         ).reset_index()
#         vessel_df.insert(0, "random_seed", seed)
#         vessel_df.insert(0, "run", i)
#         vessel_records.append(vessel_df)

#         # Repair Time at Port 

#         # Build full path to config file
#         config_path = BASE_DIR / "project" / "config" / config_name
        
#         with open(config_path, "r") as f:
#             config_data = yaml.safe_load(f)
#         port_name = config_data.get("port", None)

#         if port_name is None:
#             raise KeyError(
#                 f"'port' key not found in {config_path}, can not calculate time at port"
#             )

#         port_name = port_name.replace(".yaml", "")

#         events_df = sim.env.load_events_log_dataframe()
#         events_df["duration"] = pd.to_numeric(events_df["duration"], errors="coerce")
#         events_df = events_df.dropna(subset=["duration"])
#         df_port = events_df[events_df["agent"] == port_name]
#         total_hours = df_port["duration"].sum()
#         simulation_years = sim.env.end_year - sim.env.start_year + 1
#         avg_hours_per_year = total_hours / simulation_years
#         avg_days_per_year = avg_hours_per_year / 24
#         avg_months_per_year = avg_hours_per_year / (24 * 30.4375)

#         repair_time_records.append(
#             {
#                 "run": i,
#                 "random_seed": seed,
#                 "avg_repair_time_months": avg_months_per_year,
#                 "avg_repair_time_days": avg_days_per_year,
#             }
#         )
        
#         power_df = metrics.power_production(frequency="annual", by="windfarm", units="mwh").reset_index()
#         power_df.insert(0, "random_seed", seed)
#         power_df.insert(0, "run", i)
#         power_records.append(power_df)
        
#         fixed_df = metrics.project_fixed_costs(frequency="annual", resolution="medium").reset_index()
#         fixed_df.insert(0, "random_seed", seed)
#         fixed_df.insert(0, "run", i)
#         fixed_cost_records.append(fixed_df)
        
#         component_df = metrics.component_costs(frequency="annual", by_category=True,by_action=True).reset_index()
#         component_df.insert(0, "random_seed", seed)
#         component_df.insert(0, "run", i)
#         component_records.append(component_df)
        

#         # Cleanup logs for this simulation
#         sim.env.cleanup_log_files()

#     df_availability = pd.DataFrame(availability_records).replace([np.inf, -np.inf], np.nan).dropna()
#     df_opex = pd.concat(opex_records, ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()
#     df_vessels = pd.concat(vessel_records, ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()
#     df_repair_time = pd.DataFrame(repair_time_records).replace([np.inf, -np.inf], np.nan).dropna()
#     df_power = pd.concat(power_records, ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()
#     df_fixed = pd.concat(fixed_cost_records, ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()
#     df_component = pd.concat(component_records, ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()

#     config_prefix = config_name.replace(".yaml", "")

#     df_availability.to_csv(
#         results_dir / f"{config_prefix}_availability_results.csv",
#         index=False,
#     )
#     df_opex.to_csv(
#         results_dir / f"{config_prefix}_opex_results.csv", index=False
#     )
#     df_vessels.to_csv(
#         results_dir / f"{config_prefix}_all_vessel_results.csv", index=False
#     )
#     df_repair_time.to_csv(
#         results_dir / f"{config_prefix}_repair_time_at_port_results.csv",
#         index=False,
#     )
#     df_power.to_csv(results_dir / f"{config_prefix}_power_results.csv", index=False)
#     df_fixed.to_csv(results_dir / f"{config_prefix}_fixed_cost_results.csv", index=False)
#     df_component.to_csv(results_dir / f"{config_prefix}_component_results.csv", index=False)
    
#     print(f" All simulations complete. Results saved to {results_dir}")

In [4]:
configs = [f.name for f in Path("library/project/config").iterdir() if f.is_file() if f.name != "fixed_costs.yaml"]
configs

['SG 8.0-167 DD_base.yaml',
 'V112-3.45_base.yaml',
 'IEC-II-12MW Ref_base.yaml',
 'CSSC-H260-18_base.yaml',
 'V236-15.0 MW_base.yaml',
 'E-160-EP5_base.yaml',
 'N163-6.X_base.yaml',
 'V164-9.5 MW_base.yaml',
 'V174-9.5 MW_base.yaml',
 'IEA 15 MW Reference_base.yaml',
 'SG 11.0-200 DD_base.yaml',
 'Haliade-X 14 MW_base.yaml',
 'SWT-3.6-107_base.yaml',
 'IEC-III-10MW Ref_base.yaml',
 'SG 14-236 DD_base.yaml',
 'MySE12-242_base.yaml',
 'MySE 18.5-260_base.yaml',
 'V162-6.2 MW_base.yaml',
 'Haliade-X 12 MW_base.yaml',
 'Dongfang DEW 10MW-185_base.yaml',
 'SG 14-222 DD_base.yaml']

In [ ]:

# start_time = time()
# counter = 0 
# n_workers = cpu_count() - 1 
# print(f"Using {n_workers} processes for parallel execution")
# print(f"Starting simulations at {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")

# with ProcessPoolExecutor(max_workers=n_workers) as executor:
#     futures = {executor.submit(run_windfarm_simulations, config, RANDOM_SEEDS): config for config in configs}
#     for future in tqdm(futures, desc="Running simulations for turbine configs", total=len(configs)):
#         config = futures[future]
#         try:
#             future.result()
#             counter += 1
#         except Exception as e:
#             print(f"Error running simulations for config {config}: {e}")

# end_time = time()
# elapsed_seconds = int(end_time - start_time)

# print(f"\nCompleted {counter}/{len(configs)} configurations")
# print(f"Total time: {elapsed_seconds} seconds")
# if counter > 0:
#     print(f"Average time per config: {elapsed_seconds/counter:.0f} seconds")


Can use this following block to see total simulation results

In [8]:
def summarize_simulation(config_name,library_path=None):

    if library_path is None:
        library_path = BASE_DIR / "results"

    library_path = Path(library_path)
    if not library_path.exists():
        raise FileNotFoundError(f"The specified library_path does not exist: {library_path}")

    # Load project capacity from the config file
    config_path = BASE_DIR / "project" / "config" / config_name
    with open(config_path, "r") as f:
        config_data = yaml.safe_load(f)
    project_capacity_mw = config_data.get("project_capacity", None)

    if project_capacity_mw is None:
        raise KeyError(f"'project_capacity' key not found in {config_path}")

    capacity_kw = project_capacity_mw * 1_000
    capacity_mw = project_capacity_mw

    summary_dict = {}

    config_prefix = config_name.replace(".yaml", "")

    avail_file = library_path / f"{config_prefix}_availability_results.csv"
    opex_file = library_path / f"{config_prefix}_opex_results.csv"
    vessel_file = library_path / f"{config_prefix}_all_vessel_results.csv"
    repair_file = library_path / f"{config_prefix}_repair_time_at_port_results.csv"
    power_file = library_path / f"{config_prefix}_power_results.csv"
    fixed_file = library_path / f"{config_prefix}_fixed_cost_results.csv"
    component_file = library_path / f"{config_prefix}_component_results.csv"

    required_files = {
        f"{config_prefix}_availability_results.csv": avail_file,
        f"{config_prefix}_opex_results.csv": opex_file,
        f"{config_prefix}_all_vessel_results.csv": vessel_file,
        f"{config_prefix}_repair_time_at_port_results.csv": repair_file,
        f"{config_prefix}_power_results.csv": power_file,
        f"{config_prefix}_fixed_cost_results.csv": fixed_file,
        f"{config_prefix}_component_results.csv": component_file,
    }

    for name, path in required_files.items():
        if not path.exists():
            raise FileNotFoundError(f"Required results file not found: {path}")

    df_avail = pd.read_csv(avail_file)
    df_opex = pd.read_csv(opex_file)
    df_vessels = pd.read_csv(vessel_file)
    df_repair = pd.read_csv(repair_file)
    df_power = pd.read_csv(power_file)
    df_fixed = pd.read_csv(fixed_file)
    df_component = pd.read_csv(component_file)

    meta_cols = {"run", "random_seed", "year", "month"}

    summary_dict["Average Time-Based Availability"] = {
        "Mean": df_avail["time_based_availability"].mean() * 100,
        "Std": df_avail["time_based_availability"].std() * 100,
    }
    summary_dict["Average Production-Based Availability"] = {
        "Mean": df_avail["production_based_availability"].mean() * 100,
        "Std": df_avail["production_based_availability"].std() * 100,
    }

    opex_cols = ["operations", "port_fees", "total_labor_cost", "materials_cost"]
    for col in opex_cols:
        if col in df_opex.columns:
            pretty = {
                "operations": "Operations",
                "port_fees": "Port Fees",
                "total_labor_cost": "Labor Cost",
                "materials_cost": "Materials Cost",
            }[col]
            summary_dict[pretty] = {
                "Mean": df_opex[col].mean() / capacity_kw,
                "Std": df_opex[col].std() / capacity_kw,
            }

    vessel_cols = [c for c in df_vessels.columns if c not in meta_cols]
    vessel_total = df_vessels[vessel_cols].sum(axis=1)

    summary_dict["Equipment Cost"] = {
        "Mean": vessel_total.mean() / capacity_kw,
        "Std": vessel_total.std() / capacity_kw,
    }

    for col in vessel_cols:
        summary_dict[f"  - Vessel: {col}"] = {
            "Mean": df_vessels[col].mean() / capacity_kw,
            "Std": df_vessels[col].std() / capacity_kw,
        }

    fixed_cols = [c for c in df_fixed.columns if c not in meta_cols]
    fixed_total = pd.Series(np.zeros(len(df_fixed)), index=df_fixed.index)

    if fixed_cols:
        fixed_total = df_fixed[fixed_cols].sum(axis=1)
        summary_dict["Fixed Costs"] = {
            "Mean": fixed_total.mean() / capacity_kw,
            "Std": fixed_total.std() / capacity_kw,
        }

    power_cols = [c for c in df_power.columns if c not in meta_cols]

    preferred_power_cols = [
        c for c in power_cols
        if c.lower() in {"windfarm", "power_production", "power_production_mwh", "mwh"}
    ]

    if preferred_power_cols:
        power_series = df_power[preferred_power_cols[0]]
    else:
        numeric_power_cols = df_power[power_cols].select_dtypes(include=[np.number]).columns.tolist()
        if not numeric_power_cols:
            raise ValueError("No numeric power production columns found in power_results.csv")
        power_series = df_power[numeric_power_cols].sum(axis=1)
    power_series = power_series.replace([np.inf, -np.inf], np.nan)

    summary_dict["Annual Power Production"] = {
        "Mean": power_series.mean(),
        "Std": power_series.std(),
    }

    net_cf = power_series / (capacity_mw * 8760)
    # summary_dict["Net Capacity Factor"] = {
    #     "Mean": net_cf.mean() * 100,
    #     "Std": net_cf.std() * 100,
    # }

    component_value_col = None
    for candidate in ["value", "cost", "amount"]:
        if candidate in df_component.columns:
            component_value_col = candidate
            break

    if component_value_col is not None and "action" in df_component.columns:
        for action in ["maintenance", "repair", "delay", "travel"]:
            mask = df_component["action"].astype(str).str.lower() == action
            if mask.any():
                grouped = df_component.loc[mask].groupby(["run", "random_seed", "year"], dropna=False)[component_value_col].sum()
                summary_dict[f"Component {action.title()} Cost"] = {
                    "Mean": grouped.mean() / capacity_kw,
                    "Std": grouped.std() / capacity_kw,
                }

    base_opex_cols = [c for c in opex_cols if c in df_opex.columns]
    opex_total = df_opex[base_opex_cols].sum(axis=1) + vessel_total

    if len(fixed_total) == len(opex_total):
        opex_total = opex_total + fixed_total

    summary_dict["Total OpEx"] = {
        "Mean": opex_total.mean() / capacity_kw,
        "Std": opex_total.std() / capacity_kw,
    }

    safe_power = power_series.replace([np.inf, -np.inf], np.nan).dropna()
    safe_opex = opex_total / power_series
    # summary_dict["OpEx per MWh"] = {
    #     "Mean": safe_opex.mean(),
    #     "Std": safe_opex.std(),
    # }

    df_summary = pd.DataFrame(summary_dict).T

    preferred_order = [
        "Average Time-Based Availability",
        "Average Production-Based Availability",
        "Net Capacity Factor",
        "Annual Power Production",
        "Average Repair Time At Port",
        "Operations",
        "Port Fees",
        "Labor Cost",
        "Materials Cost",
        "Fixed Costs",
        "Equipment Cost",
        *[f"  - Vessel: {v}" for v in vessel_cols],
        "Component Maintenance Cost",
        "Component Repair Cost",
        "Component Delay Cost",
        "Component Travel Cost",
        "Total OpEx",
        # "OpEx per MWh",
    ]

    ordered_rows = [r for r in preferred_order if r in df_summary.index]
    remaining_rows = [r for r in df_summary.index if r not in ordered_rows]
    df_summary = df_summary.loc[ordered_rows + remaining_rows]

    units = []
    for idx in df_summary.index:
        idx_lower = idx.lower()
        if "availability" in idx_lower or "capacity factor" in idx_lower:
            units.append("%")
        elif "power production" in idx_lower:
            units.append("MWh / yr")
        elif "repair time at port" in idx_lower:
            units.append("months / yr")
        elif "per mwh" in idx_lower:
            units.append("$ / MWh")
        else:
            units.append("$ / kW-yr")

    df_summary.insert(0, "Units", units)

    df_summary.replace(0, np.nan, inplace=True)
    df_summary.dropna(how="all", inplace=True)

    return df_summary

In [ ]:

# df_summary = summarize_simulation(config_name=configs[0])
# df_summary.style.format(na_rep="-", precision=1) 




In [6]:
turbine_summaries = []
for config in configs:
    try:
        summary = summarize_simulation(config_name=config)
        turbine_summaries.append((config.replace(".yaml", ""), summary))
    except Exception as e:
        print(f"Skipping {config}: {str(e)}")
        continue
    
comparison_data = []
for turbine, df_summary in turbine_summaries:
    summary_dict = df_summary["Mean"].to_dict()  
    summary_dict["Turbine"] = turbine  
    comparison_data.append(summary_dict)

df_comparison = pd.DataFrame(comparison_data)

columns = ["Turbine"] + [col for col in df_comparison.columns if col != "Turbine"]
df_comparison = df_comparison[columns]

In [7]:
df_comparison.style.format(na_rep="-", precision=1) 

,Turbine,Average Time-Based Availability,Average Production-Based Availability,Annual Power Production,Operations,Port Fees,Labor Cost,Materials Cost,Fixed Costs,Equipment Cost,- Vessel: Cable Laying Vessel,- Vessel: Crew Transfer Vessel,- Vessel: Diving Support Vessel,- Vessel: Heavy Lift Vessel,Total OpEx,OpEx per MWh
0,SG 8.0-167 DD_base,94.6,468.8,4653888.3,52.5,4.2,-,34.2,52.5,101.2,50.6,2.4,2.0,46.1,244.5,90.4
1,V112-3.45_base,96.8,478.8,606330.6,52.5,9.5,-,70.9,52.5,230.6,112.1,5.6,4.7,108.3,415.9,149.5
2,IEC-II-12MW Ref_base,94.3,93.2,3144313.1,52.5,2.5,-,22.8,52.5,66.7,32.6,1.5,1.3,31.3,196.9,50.9
3,CSSC-H260-18_base,96.4,3358.7,8223373.6,52.5,2.5,-,20.2,52.5,51.4,27.1,1.5,1.0,21.8,179.1,69.5
4,V236-15.0 MW_base,95.9,127822.2,44309932.5,52.5,2.5,-,21.6,52.5,58.6,29.8,1.5,1.1,26.2,187.6,48.5
5,E-160-EP5_base,95.9,3268.2,8036188.0,52.5,6.3,-,46.4,52.5,148.5,72.5,3.7,2.9,69.5,306.2,90.1
6,N163-6.X_base,96.7,461625.0,3084094169.7,52.5,4.9,-,40.5,52.5,120.8,60.8,2.9,2.3,54.9,271.1,91.5
7,V164-9.5 MW_base,94.5,6732.8,41429399.5,52.5,3.6,-,29.5,52.5,84.9,42.5,2.1,1.7,38.6,222.9,96.7
8,V174-9.5 MW_base,95.8,5183.4,38851901.0,52.5,3.5,-,27.0,52.5,83.3,40.0,2.0,1.7,39.5,218.6,65.4
9,IEA 15 MW Reference_base,95.1,12567797.5,78830706.1,52.5,2.5,-,22.3,52.5,59.5,31.5,1.5,1.1,25.5,189.3,46.0


In [10]:
df_comparison.to_csv("turbines_results.csv")


In [ ]:
# with storm effects
target_config = "IEA 15 MW Reference_base.yaml"
df_one = summarize_simulation(config_name=target_config)
display(df_one.style.format(na_rep="-", precision=2))

,Units,Mean,Std
Average Time-Based Availability,%,78.74,25.09
Average Production-Based Availability,%,50.66,60.92
Annual Power Production,MWh / yr,1844250.80,1696920.16
Operations,$ / kW-yr,52.46,0.06
Port Fees,$ / kW-yr,2.52,-
Labor Cost,$ / kW-yr,-,-
Materials Cost,$ / kW-yr,23.11,12.02
Fixed Costs,$ / kW-yr,52.46,0.06
Equipment Cost,$ / kW-yr,57.53,24.30
- Vessel: Cable Laying Vessel,$ / kW-yr,28.84,19.80


In [10]:
target_config = "IEA 15 MW Reference_base.yaml"
df_one = summarize_simulation(config_name=target_config)
display(df_one.style.format(na_rep="-", precision=2))

,Units,Mean,Std
Average Time-Based Availability,%,95.05,2.73
Average Production-Based Availability,%,93.17,-
Annual Power Production,MWh / yr,3184952.19,413810.04
Operations,$ / kW-yr,52.46,0.06
Port Fees,$ / kW-yr,2.52,-
Labor Cost,$ / kW-yr,-,-
Materials Cost,$ / kW-yr,20.90,13.13
Fixed Costs,$ / kW-yr,52.46,0.06
Equipment Cost,$ / kW-yr,57.82,24.53
- Vessel: Cable Laying Vessel,$ / kW-yr,29.27,20.08
